# Chapter 25 — Hyperparameter Tuning and Honest Model Selection

*From Absolute Zero* — companion notebook.

Every block below is the code printed in the chapter, in the same order. Run the cells top to bottom; the output should match the book exactly. If it does not, check `requirements.txt` first, then `docs/TROUBLESHOOTING.md`.

In [1]:
!pip -q install -r https://raw.githubusercontent.com/FromAbsoluteZero/CodeBase/main/requirements.txt  # Colab only; skip locally

zsh:1: command not found: pip


## Create the data

Run once. Every dataset in this book is generated by code you can read — nothing is downloaded, so nothing can rot behind a dead link. This is the printed block from Chapter 24 (`code/ch24/gen_orders.py` in the repository).

In [2]:
import numpy as np, pandas as pd
rng = np.random.default_rng(24)
n = 8000

city = rng.choice([f"CITY_{i:03d}" for i in range(180)], n)      # high cardinality
plan = rng.choice(["basic", "plus", "pro"], n, p=[.55, .32, .13])
channel = rng.choice(["web", "app", "phone"], n, p=[.5, .38, .12])
signup = pd.to_datetime("2023-01-01") + pd.to_timedelta(
    rng.integers(0, 730, n), unit="D")
income = np.round(np.exp(rng.normal(10.2, 0.55, n)), 0)
sessions = rng.poisson(6, n)
basket = np.round(np.exp(rng.normal(3.1, 0.7, n)), 2)

# churn depends on plan, engagement, and value-for-money -- not on city
z = (-0.4
     - 0.55 * (plan == "pro") + 0.35 * (plan == "basic")
     - 0.09 * sessions
     + 1.85 * (basket / (income / 1000) > 1.4)
     + 0.30 * (channel == "phone")
     + rng.normal(0, 0.6, n))
churn = (rng.random(n) < 1 / (1 + np.exp(-z))).astype(int)

df = pd.DataFrame({"City": city, "Plan": plan, "Channel": channel,
                   "SignupDate": signup.strftime("%Y-%m-%d"),
                   "AnnualIncome": income, "Sessions": sessions,
                   "AvgBasket": basket, "Churn": churn})
df.loc[rng.random(n) < 0.09, "AnnualIncome"] = np.nan     # real gaps
df.to_csv("customers.csv", index=False)
print(f"wrote customers.csv: {n:,} customers, churn {churn.mean():.1%}, "
      f"{df.City.nunique()} cities, {df.AnnualIncome.isna().sum()} missing incomes")

wrote customers.csv: 8,000 customers, churn 44.8%, 180 cities, 720 missing incomes


## Shared setup

Imports and the objects the blocks below reuse. The chapter prints these once and then continues the same session. This cell is `code/ch25/_lib.py`.

In [3]:
import numpy as np, pandas as pd, warnings; warnings.filterwarnings("ignore")
from scipy.stats import loguniform, randint
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import (GridSearchCV, RandomizedSearchCV,
    cross_val_score, StratifiedKFold, train_test_split)
from sklearn.datasets import make_classification
# customers.csv is created by Chapter 24 (code/ch24/gen_orders.py). The blocks read it from the
# working directory exactly as the book does; if it is not here yet, use the copy shipped in
# data/generated/ (byte-identical to what the generator writes).
import os as _os, shutil as _shutil
if not _os.path.exists("customers.csv"):
    for _d in ("../../data/generated", "../data/generated", "data/generated"):
        if _os.path.exists(_os.path.join(_d, "customers.csv")):
            _shutil.copy(_os.path.join(_d, "customers.csv"), "customers.csv"); break
df = pd.read_csv("customers.csv", parse_dates=["SignupDate"])
y = df.pop("Churn").values
df = df.drop(columns=["City", "SignupDate"])
NUM = ["AnnualIncome", "Sessions", "AvgBasket"]
CAT = ["Plan", "Channel"]
def base_pipe(clf):
    num = Pipeline([("imp", SimpleImputer(strategy="median")),
                    ("sc", StandardScaler())])
    return Pipeline([("pre", ColumnTransformer([
                        ("n", num, NUM),
                        ("c", OneHotEncoder(handle_unknown="ignore"), CAT)])),
                     ("clf", clf)])
cv = StratifiedKFold(5, shuffle=True, random_state=0)

## The chapter code

### Block 1  (`c1.py`)

In [4]:
# Grid search: every combination, exhaustively.
grid = {"clf__C": [0.01, 0.1, 1, 10, 100],
        "clf__penalty": ["l1", "l2"]}
gs = GridSearchCV(base_pipe(LogisticRegression(max_iter=4000, random_state=0,
                            solver="liblinear")),
                  grid, cv=cv, scoring="roc_auc", n_jobs=-1)
gs.fit(df, y)
print(f"fits: 10 combinations x 5 folds = 50")
print(f"best params: {gs.best_params_}")
print(f"best CV AUC: {gs.best_score_:.4f}")

r = pd.DataFrame(gs.cv_results_).nlargest(4, "mean_test_score")
print(f"\n{'C':>8}{'penalty':>9}{'mean':>9}{'sd':>8}")
for _, row in r.iterrows():
    print(f"{row['param_clf__C']:>8}{row['param_clf__penalty']:>9}"
          f"{row['mean_test_score']:>9.4f}{row['std_test_score']:>8.4f}")

fits: 10 combinations x 5 folds = 50
best params: {'clf__C': 0.01, 'clf__penalty': 'l2'}
best CV AUC: 0.6716

       C  penalty     mean      sd
    0.01       l2   0.6716  0.0161
     0.1       l1   0.6713  0.0165
     0.1       l2   0.6712  0.0164
   100.0       l1   0.6711  0.0165


### Block 2  (`c2.py`)

In [5]:
# Random search covers a wider space for the same budget, and samples
# continuous parameters on the scale they actually vary on.
space = {"clf__learning_rate": loguniform(1e-3, 3e-1),
         "clf__max_depth": randint(2, 9),
         "clf__max_iter": randint(60, 400),
         "clf__l2_regularization": loguniform(1e-3, 1e1)}
rs = RandomizedSearchCV(base_pipe(HistGradientBoostingClassifier(
                            random_state=0)),
                        space, n_iter=30, cv=cv, scoring="roc_auc",
                        random_state=0, n_jobs=-1)
rs.fit(df, y)
print(f"30 samples x 5 folds = 150 fits")
for k, v in rs.best_params_.items():
    print(f"  {k.replace('clf__',''):<20} "
          f"{v if isinstance(v, int) else round(v, 5)}")
print(f"best CV AUC: {rs.best_score_:.4f}")

r = pd.DataFrame(rs.cv_results_).nlargest(5, "mean_test_score")
print(f"\ntop five of thirty, mean +/- sd")
for _, row in r.iterrows():
    print(f"  {row['mean_test_score']:.4f} +/- {row['std_test_score']:.4f}")

30 samples x 5 folds = 150 fits
  l2_regularization    0.01145
  learning_rate        0.00967
  max_depth            5
  max_iter             244
best CV AUC: 0.6870

top five of thirty, mean +/- sd
  0.6870 +/- 0.0143
  0.6862 +/- 0.0123
  0.6860 +/- 0.0120
  0.6852 +/- 0.0119
  0.6851 +/- 0.0117


### Block 3  (`c3.py`)

In [6]:
# The tuned score is the maximum of many noisy estimates, so it is biased
# upward -- and the bias grows with how many candidates you try.
X, yy = make_classification(n_samples=600, n_features=20, n_informative=6,
                            flip_y=0.15, random_state=0)
space = {"learning_rate": loguniform(1e-3, 3e-1),
         "max_depth": randint(2, 8), "max_iter": randint(60, 300),
         "l2_regularization": loguniform(1e-3, 1e1)}
outer = StratifiedKFold(4, shuffle=True, random_state=0)

print(f"{'candidates':>11}{'inner (tuned)':>15}{'nested (honest)':>17}"
      f"{'optimism':>10}")
for n_iter in (5, 20, 60):
    s = RandomizedSearchCV(HistGradientBoostingClassifier(random_state=0),
                           space, n_iter=n_iter, cv=3, random_state=0,
                           n_jobs=-1)
    s.fit(X, yy)
    nested = cross_val_score(s, X, yy, cv=outer, n_jobs=-1).mean()
    print(f"{n_iter:>11}{s.best_score_:>15.4f}{nested:>17.4f}"
          f"{s.best_score_ - nested:>+10.4f}")

 candidates  inner (tuned)  nested (honest)  optimism


          5         0.8183           0.8233   -0.0050


         20         0.8300           0.8200   +0.0100


         60         0.8300           0.8083   +0.0217


### Block 4  (`c4.py`)

In [7]:
# Preprocessing choices are hyperparameters too. Tune them in the same
# search, so each option is evaluated with the folds refitted around it.
space = {
    "pre__n__imp__strategy": ["median", "mean"],
    "clf__learning_rate": loguniform(1e-3, 3e-1),
    "clf__max_depth": randint(2, 9),
    "clf__max_iter": randint(60, 400),
    "clf__l2_regularization": loguniform(1e-3, 1e1),
}
rs = RandomizedSearchCV(base_pipe(HistGradientBoostingClassifier(
                            random_state=0)),
                        space, n_iter=40, cv=cv, scoring="roc_auc",
                        random_state=0, n_jobs=-1)
rs.fit(df, y)
print(f"best CV AUC {rs.best_score_:.4f}")
print(f"imputation chosen: {rs.best_params_['pre__n__imp__strategy']}")

r = pd.DataFrame(rs.cv_results_)
print(f"\n{'imputation':>12}{'best of that option':>22}{'tried':>8}")
for s in ("median", "mean"):
    sub = r[r["param_pre__n__imp__strategy"] == s]
    print(f"{s:>12}{sub['mean_test_score'].max():>22.4f}{len(sub):>8}")

best CV AUC 0.6876
imputation chosen: median

  imputation   best of that option   tried
      median                0.6876      19
        mean                0.6871      21


### Block 5  (`c5.py`)

In [8]:
# The honest final report: tune on training, estimate honestly, then
# open the test set exactly once.
from sklearn.metrics import roc_auc_score
Xtr, Xte, ytr, yte = train_test_split(df, y, test_size=0.3,
                                      random_state=0, stratify=y)

space = {"clf__learning_rate": loguniform(1e-3, 3e-1),
         "clf__max_depth": randint(2, 9),
         "clf__max_iter": randint(60, 400),
         "clf__l2_regularization": loguniform(1e-3, 1e1)}
search = RandomizedSearchCV(base_pipe(HistGradientBoostingClassifier(
                                random_state=0)),
                            space, n_iter=40, cv=cv, scoring="roc_auc",
                            random_state=0, n_jobs=-1)
search.fit(Xtr, ytr)

nested = cross_val_score(search, Xtr, ytr,
                         cv=StratifiedKFold(4, shuffle=True, random_state=1),
                         scoring="roc_auc", n_jobs=-1).mean()
test = roc_auc_score(yte, search.predict_proba(Xte)[:, 1])
baseline = cross_val_score(base_pipe(LogisticRegression(max_iter=4000)),
                           Xtr, ytr, cv=cv, scoring="roc_auc").mean()

print(f"untuned logistic baseline (CV)  {baseline:.4f}")
print(f"tuned inner CV score            {search.best_score_:.4f}")
print(f"nested CV estimate              {nested:.4f}")
print(f"test set, opened once           {test:.4f}")
print(f"\ntuning bought {search.best_score_ - baseline:+.4f} on "
      f"the inner score")
print(f"and {nested - baseline:+.4f} once measured honestly")

untuned logistic baseline (CV)  0.6802
tuned inner CV score            0.6895
nested CV estimate              0.6865
test set, opened once           0.6764

tuning bought +0.0093 on the inner score
and +0.0063 once measured honestly
